#  Closed Deals - Bronze Ingestion


## Imports

In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, TimestampType, BooleanType, StructField, DecimalType, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_closed_deals"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist_marketing"
source_dataset = "closed_deals"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("mql_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("sdr_id", StringType(), True),
    StructField("sr_id", StringType(), True),
    StructField("won_date", TimestampType(), True),
    StructField("business_segment", StringType(), True),
    StructField("lead_type", StringType(), True),
    StructField("lead_behaviour_profile", StringType(), True),
    StructField("has_company", BooleanType(), True),
    StructField("has_gtin", BooleanType(), True),
    StructField("average_stock", StringType(), True),
    StructField("business_type", StringType(), True),
    StructField("declared_product_catalog_size", DecimalType(10, 1), True),
    StructField("declared_monthly_revenue", DecimalType(18, 2), True)
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- mql_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- sdr_id: string (nullable = true)
 |-- sr_id: string (nullable = true)
 |-- won_date: timestamp (nullable = true)
 |-- business_segment: string (nullable = true)
 |-- lead_type: string (nullable = true)
 |-- lead_behaviour_profile: string (nullable = true)
 |-- has_company: boolean (nullable = true)
 |-- has_gtin: boolean (nullable = true)
 |-- average_stock: string (nullable = true)
 |-- business_type: string (nullable = true)
 |-- declared_product_catalog_size: decimal(10,1) (nullable = true)
 |-- declared_monthly_revenue: decimal(18,2) (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)

In [0]:
display(spark.table(target_table).limit(5))

mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26T19:58:54.000Z,pet,online_medium,cat,null,null,null,reseller,null,0.00,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/closed_deals/olist_closed_deals_dataset.csv,2026-08-02T21:45:38.000Z,2026-08-03T01:43:34.590Z,cc7d51c8-a726-4bd8-8263-bfa64c6a89d6,olist_marketing,closed_deals
a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08T20:17:59.000Z,car_accessories,industry,eagle,null,null,null,reseller,null,0.00,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/closed_deals/olist_closed_deals_dataset.csv,2026-08-02T21:45:38.000Z,2026-08-03T01:43:34.590Z,cc7d51c8-a726-4bd8-8263-bfa64c6a89d6,olist_marketing,closed_deals
327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05T17:27:23.000Z,home_appliances,online_big,cat,null,null,null,reseller,null,0.00,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/closed_deals/olist_closed_deals_dataset.csv,2026-08-02T21:45:38.000Z,2026-08-03T01:43:34.590Z,cc7d51c8-a726-4bd8-8263-bfa64c6a89d6,olist_marketing,closed_deals
f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17T13:51:03.000Z,food_drink,online_small,null,null,null,null,reseller,null,0.00,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/closed_deals/olist_closed_deals_dataset.csv,2026-08-02T21:45:38.000Z,2026-08-03T01:43:34.590Z,cc7d51c8-a726-4bd8-8263-bfa64c6a89d6,olist_marketing,closed_deals
ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03T20:17:45.000Z,home_appliances,industry,wolf,null,null,null,manufacturer,null,0.00,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/closed_deals/olist_closed_deals_dataset.csv,2026-08-02T21:45:38.000Z,2026-08-03T01:43:34.590Z,cc7d51c8-a726-4bd8-8263-bfa64c6a89d6,olist_marketing,closed_deals


In [0]:
spark.table(target_table).count()

842